### [ 주제: 붓꽃의 품종 분류 ] <hr>

- 학습 종류: 지도 학습
- 학습 방법: 분류 관련 모델 중에서 선택
- 데이터: iris.csv
- 구현 단계
    * (1) 데이터 준비 및 확인
    * (2) 데이터 전처리
    * (3) 학습/ 검증/ 테스트용 데이터셋 분리
    * (4) 학습 => 교차검증+ 하이퍼파라미터 튜닝
    * (5) 평가

- **[0] 모듈 로딩**

In [5]:
## -------------------------------------
## 이미지 데이터 로딩 & 기본 정보
## -------------------------------------
import cv2
import os
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
import matplotlib.cm as cm
import sys
sys.path.append(r"C:\KDT14\7_CV")

import importlib
import cv_utils

importlib.reload(cv_utils)

from sklearn.model_selection import train_test_split #학습용| 테스트용 데이터 분할
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier # 모델클래스
from sklearn.metrics import accuracy_score, classification_report # 평가
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV 

- **[1] 데이터 준비 및 확인**

In [6]:
DATA_FILE= './iris.csv'

dataDF= pd.read_csv(DATA_FILE)

In [7]:
## 데이터 기본 정보 확인=> 형태, 컬럼(타입, 결측치 여부, 개수), 실제 데이터와 비교, 컬럼별 통계치
dataDF.info()
display(dataDF.head(3))
dataDF.describe() # 통계치
dataDF.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal.length  150 non-null    float64
 1   sepal.width   150 non-null    float64
 2   petal.length  150 non-null    float64
 3   petal.width   150 non-null    float64
 4   variety       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa


,sepal.length,sepal.width,petal.length,petal.width,variety
count,150.000000,150.000000,150.000000,150.000000,150
unique,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,Setosa
freq,NaN,NaN,NaN,NaN,50
mean,5.843333,3.057333,3.758000,1.199333,NaN
std,0.828066,0.435866,1.765298,0.762238,NaN
min,4.300000,2.000000,1.000000,0.100000,NaN
25%,5.100000,2.800000,1.600000,0.300000,NaN
50%,5.800000,3.000000,4.350000,1.300000,NaN
75%,6.400000,3.300000,5.100000,1.800000,NaN


In [8]:
## [ 1차 해석 ]
## - variety 컬럼: str==> category 형변환
## - 결측치: 없음
## - 타겟/ 라벨/ 클래스 컬럼: variety
## - 피쳐/ 특성/ 속성 컬럼: sepal/length, sepal.width, petal.length, petal.width

- **[2]데이터 전처리**

In [9]:
## variety 컬럼 : str == > category 형변환
## - df.astype으로 자료형 변환: inplace 매개변수x. 반드시 저장

dataDF.variety= dataDF.variety.astype('category')

display(dataDF.dtypes)

sepal.length     float64
sepal.width      float64
petal.length     float64
petal.width      float64
variety         category
dtype: object

- **[3]학습용/ 검증용/ 테스트용 데이터셋 준비**

In [10]:
## 피처와 타겟 분리
featureDF= dataDF[dataDF.columns[:-1]]
targetSR= dataDF[dataDF.columns[-1]]

print(f'피처: {featureDF.shape}, 타겟: {targetSR.shape}')

피처: (150, 4), 타겟: (150,)


In [11]:
## => 학습용 |테스트용 분리 : 분류 => 타겟 클래스 비율 유지 stratify 매개변수 설정
#                          재현성 => random_state 매개변수 설정
#                          피처 차원-> 2차원
#                          타겟 차원-> 1차원



X_train, X_test, y_train, y_test= train_test_split(featureDF, targetSR, test_size=0.2, random_state= 10, stratify= targetSR)

In [12]:
## => 확인
print(f'학습용 : {X_train.shape}, {y_train.shape} 테스트용 : {X_test.shape}, {y_test.shape}')
print(f'클래스 데이터 구성\n', y_train.value_counts().tolist())
print(f'\n테스트 데이터 구성\n', y_test.value_counts().tolist())


학습용 : (120, 4), (120,) 테스트용 : (30, 4), (30,)
클래스 데이터 구성
 [40, 40, 40]

테스트 데이터 구성
 [10, 10, 10]


- **[4] 교차검증+ 튜닝**

In [13]:
## ------------------------------------------
## => GridSearchCV 인스턴스 생성 및 진행
## - 모든 조합의 모델 생성
## - 조합된 모델별 교차 검증 진행
## - 최적 조합의 하이퍼파라미터 추출
## - 단점 : 시간이 오래 걸림 !!!
## ------------------------------------------
## => 모델인스턴스 : KNN
kModel = KNeighborsClassifier()

## => 모델의 하이퍼파라미터 Dict
param_dict ={'n_neighbors' : range(1,51,2),
'p': [1,2],
'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']}

StratifiedKFold(shuffle=True, random_state= 10)

## => 튜닝 및 교차검증 진행 인스턴스
gsModel = GridSearchCV(kModel, param_grid=param_dict, return_train_score=True)


In [14]:
## 하이퍼파라미터 조합으로 모델 생성 및 교차검증 학습 진행
## => Y_TRAIN에 따라서 KFold, StatifiedFold로 선택됨
gsModel.fit(X_train, y_train)

GridSearchCV(estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                         'n_neighbors': range(1, 51, 2), 'p': [1, 2]},
             return_train_score=True)

In [15]:
## 학습 후 설정된 모델 파라미터 확인
gsModel.cv_results_

{'mean_fit_time': array([0.00134926, 0.00140781, 0.0016068 , 0.00160799, 0.00191665,
        0.00231447, 0.00161433, 0.00100327, 0.00170455, 0.00160728,
        0.00181041, 0.00172567, 0.00187683, 0.00140634, 0.00160637,
        0.00130506, 0.00180531, 0.00202274, 0.00091481, 0.00201621,
        0.00120945, 0.00151501, 0.00130463, 0.00212097, 0.0016633 ,
        0.00137634, 0.00150714, 0.00150828, 0.00171251, 0.00160718,
        0.00170584, 0.00183358, 0.00140433, 0.00181017, 0.001407  ,
        0.00158911, 0.0016892 , 0.00147328, 0.00125661, 0.00130272,
        0.00169029, 0.00130091, 0.00151224, 0.00150657, 0.00127316,
        0.0015192 , 0.00130596, 0.00150876, 0.00123963, 0.00141959,
        0.00171247, 0.00115643, 0.00143299, 0.00110893, 0.00169334,
        0.00159554, 0.00160346, 0.0014308 , 0.00131607, 0.00171456,
        0.00181208, 0.00127854, 0.0016089 , 0.00221257, 0.00150046,
        0.00132208, 0.00170808, 0.00160408, 0.00128665, 0.00164371,
        0.00100856, 0.0014421 ,

In [16]:
## => 학습 후 설정된 모델 파라미터 확인
## - cv_results_ : 각 모델의 교차검증 결과 저장 dict 형식
cv_resultDF = pd.DataFrame(gsModel.cv_results_)

## - best_params_ : 최고 성능의 하이퍼파라미터 조합
print(f'gsModel.best_params _: {gsModel.best_params_}=> {gsModel.best_score_:.4f}')

bestModel= gsModel.best_estimator_

gsModel.best_params _: {'algorithm': 'auto', 'n_neighbors': 15, 'p': 1}=> 0.9667


- **[5] 시각화**

In [17]:

## => 모델 조합별 교차검증 결과
cv_resultDF.info()

# select_cols = [ 'param_algorithm', 'param_n_neighbors', 'param_p',
# 'split0_test_score', 'split1_test_score', 'split2_test_score',
# 'split3_test_score', 'split4_test_score', 'mean_test_score',
# 'std_test_score', 'rank_test_score']

# cv_resultDF[select_cols].head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   mean_fit_time       200 non-null    float64
 1   std_fit_time        200 non-null    float64
 2   mean_score_time     200 non-null    float64
 3   std_score_time      200 non-null    float64
 4   param_algorithm     200 non-null    object 
 5   param_n_neighbors   200 non-null    int64  
 6   param_p             200 non-null    int64  
 7   params              200 non-null    object 
 8   split0_test_score   200 non-null    float64
 9   split1_test_score   200 non-null    float64
 10  split2_test_score   200 non-null    float64
 11  split3_test_score   200 non-null    float64
 12  split4_test_score   200 non-null    float64
 13  mean_test_score     200 non-null    float64
 14  std_test_score      200 non-null    float64
 15  rank_test_score     200 non-null    int32  
 16  split0_t

In [18]:

## => 모델 조합별 교차검증 결과
cv_resultDF.info()

## => 확인하고 싶은 컬럼만 선택
cols =['rank_test_score', 'param_algorithm', 'param_n_neighbors', 'param_p', 'mean_test_score']

display(cv_resultDF[cols])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   mean_fit_time       200 non-null    float64
 1   std_fit_time        200 non-null    float64
 2   mean_score_time     200 non-null    float64
 3   std_score_time      200 non-null    float64
 4   param_algorithm     200 non-null    object 
 5   param_n_neighbors   200 non-null    int64  
 6   param_p             200 non-null    int64  
 7   params              200 non-null    object 
 8   split0_test_score   200 non-null    float64
 9   split1_test_score   200 non-null    float64
 10  split2_test_score   200 non-null    float64
 11  split3_test_score   200 non-null    float64
 12  split4_test_score   200 non-null    float64
 13  mean_test_score     200 non-null    float64
 14  std_test_score      200 non-null    float64
 15  rank_test_score     200 non-null    int32  
 16  split0_t

,rank_test_score,param_algorithm,param_n_neighbors,param_p,mean_test_score
0,29,auto,1,1,0.950000
1,29,auto,1,2,0.950000
2,62,auto,3,1,0.941667
3,62,auto,3,2,0.941667
4,85,auto,5,1,0.933333
...,...,...,...,...,...
195,143,brute,45,2,0.908333
196,182,brute,47,1,0.900000
197,143,brute,47,2,0.908333
198,143,brute,49,1,0.908333


In [19]:
param_dict ={'n_neighbors' : range(1,51,2), 'p':[1,2], 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']}

## => 튜닝 및 교차검증 진행 인스턴스
gsModel = GridSearchCV(kModel, param_grid=param_dict, return_train_score=True)

In [20]:
gsModel.fit(X_train, y_train)

GridSearchCV(estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                         'n_neighbors': range(1, 51, 2), 'p': [1, 2]},
             return_train_score=True)

In [21]:
cv_resultDF.info()

cols =['rank_test_score', 'param_algorithm', 'param_n_neighbors', 'param_p', 'mean_test_score']

selDF= cv_resultDF[cols]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   mean_fit_time       200 non-null    float64
 1   std_fit_time        200 non-null    float64
 2   mean_score_time     200 non-null    float64
 3   std_score_time      200 non-null    float64
 4   param_algorithm     200 non-null    object 
 5   param_n_neighbors   200 non-null    int64  
 6   param_p             200 non-null    int64  
 7   params              200 non-null    object 
 8   split0_test_score   200 non-null    float64
 9   split1_test_score   200 non-null    float64
 10  split2_test_score   200 non-null    float64
 11  split3_test_score   200 non-null    float64
 12  split4_test_score   200 non-null    float64
 13  mean_test_score     200 non-null    float64
 14  std_test_score      200 non-null    float64
 15  rank_test_score     200 non-null    int32  
 16  split0_t

In [22]:
selDF.sort_values(by= ['rank_test_score'])
cv_resultDF.iloc[14]

mean_fit_time                                                 0.001606
std_fit_time                                                  0.000589
mean_score_time                                               0.001913
std_score_time                                                0.000371
param_algorithm                                                   auto
param_n_neighbors                                                   15
param_p                                                              1
params                {'algorithm': 'auto', 'n_neighbors': 15, 'p': 1}
split0_test_score                                             0.958333
split1_test_score                                             0.916667
split2_test_score                                                  1.0
split3_test_score                                             0.958333
split4_test_score                                                  1.0
mean_test_score                                               0.966667
std_te

In [23]:
## ------------------------------------------
## => RandomizedSearchCV 인스턴스 생성 및 진행
## - GridSearchCV 보다 빠르지만 성능이 안 좋을수도 있음
## - 지정된 개수 만큼의 모델을 조합 => 속도 빠름
## - 조합된 모델별 교차 검증 진행
## - 단점 : GridSearchCV보다 최적의 조합 아닐 수 도 있음
## cv 파라미터
##   * 기본값 : 5
##   * 학습 시 전달하는 y값에 따라서 KFold, StratifiedKFold 설정
## - random_state 파라미터
##   * 재현성 위해서 설정
## - n_iter 파라미터
##   * 무작위 조합할 모델 개수 설정
## ------------------------------------------


## => 교차검증 인스턴스 생성
skFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

## => 튜닝 및 교차검증 진행 인스턴스
rsModel = RandomizedSearchCV(kModel, param_distributions=param_dict, cv=skFold, n_iter=50, random_state= 10, return_train_score=True)



In [24]:
## 교차검증 진행
rsModel.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=10, shuffle=True),
                   estimator=KNeighborsClassifier(), n_iter=50,
                   param_distributions={'algorithm': ['auto', 'ball_tree',
                                                      'kd_tree', 'brute'],
                                        'n_neighbors': range(1, 51, 2),
                                        'p': [1, 2]},
                   random_state=10, return_train_score=True)

In [25]:
print(f'최고조합: {rsModel.best_params_} , 최고점수: {rsModel.best_score_}')

## 최고 조합으로 재학습된 모델 인스턴스
rs_best_model= rsModel.best_estimator_

## 조합 모델 별 학습 성능
cv_resultDF= pd.DataFrame(rsModel.cv_results_)
cv_resultDF.info()

최고조합: {'p': 2, 'n_neighbors': 9, 'algorithm': 'brute'} , 최고점수: 0.9583333333333334
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   mean_fit_time       50 non-null     float64
 1   std_fit_time        50 non-null     float64
 2   mean_score_time     50 non-null     float64
 3   std_score_time      50 non-null     float64
 4   param_p             50 non-null     int64  
 5   param_n_neighbors   50 non-null     int64  
 6   param_algorithm     50 non-null     object 
 7   params              50 non-null     object 
 8   split0_test_score   50 non-null     float64
 9   split1_test_score   50 non-null     float64
 10  split2_test_score   50 non-null     float64
 11  split3_test_score   50 non-null     float64
 12  split4_test_score   50 non-null     float64
 13  mean_test_score     50 non-null     float64
 14  std_test_score      50 non

### [ 과제 ] <hr>

In [29]:
## 사용자로부터 입력받은 값 예측
in_data = input("붓꽃 정보 입력(예: 0.12 0.08 1.23 1.44):").split(" ")
newDF = pd.DataFrame([in_data], columns=X_train.columns)
print(newDF)


gs_best_model = gsModel.best_estimator_
## 예측 진행
y_pre = gs_best_model.predict(newDF)

print(y_pre)

  sepal.length sepal.width petal.length petal.width
0         0.12        0.08         1.23        1.44
['Setosa']
